# Importy

In [2]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import time
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import os

# Za 4 punkty

### Opis środowiska Lunar Lander

...

### Opis algorytmu Proximal Policy Optimization (PPO)

...

### Stałe

In [ ]:
env_id = "LunarLander-v3"
total_timesteps = 150000
n_runs = 10

### Zestawy parametrów

In [ ]:
hyperparams_sets = [
    {"name": "Set 1 (Default)", "lr": 3e-4, "batch_size": 64},
    {"name": "Set 2 (Fast/Large)", "lr": 1e-3, "batch_size": 128},
    {"name": "Set 3 (Slow/Small)", "lr": 5e-5, "batch_size": 32}
]

### Trening dla różnych zestawów hiperparametrów

In [ ]:
log_dir = "./logs"
os.makedirs(log_dir, exist_ok=True)

results = {}
execution_times = []

for hp in hyperparams_sets:
    print(f"Starting: {hp['name']}")
    all_runs_rewards = []
    
    for run in range(n_runs):
        env = gym.make(env_id, continuous=True)
        
        run_log_dir = f"{log_dir}/{hp['name']}_run_{run}"
        os.makedirs(run_log_dir, exist_ok=True)
        env = Monitor(env, run_log_dir)
        
        model = PPO(
            "MlpPolicy", 
            env, 
            learning_rate=hp["lr"], 
            batch_size=hp["batch_size"], 
            seed=run, 
            verbose=0
        )
        
        start_time = time.time()
        model.learn(total_timesteps=total_timesteps)
        model.save(f"ppo_lunar_{hp['name']}_run_{run}")
        end_time = time.time()
        
        execution_times.append((end_time - start_time) / total_timesteps)
        
        x, y = ts2xy(load_results(run_log_dir), "timesteps")
        
        eval_steps = np.linspace(0, total_timesteps, 100)
        y_interp = np.interp(eval_steps, x, y)
        all_runs_rewards.append(y_interp)
        
        env.close()
        
    results[hp["name"]] = {
        "steps": eval_steps,
        "mean": np.mean(all_runs_rewards, axis=0),
        "std": np.std(all_runs_rewards, axis=0)
    }

### Wizualizacja procesu uczenia

In [ ]:
avg_step_time = np.mean(execution_times)
print(f"Average time per timestep: {avg_step_time:.6f} seconds")

plt.figure(figsize=(10, 6))
for name, data in results.items():
    plt.plot(data["steps"], data["mean"], label=f"{name} (Mean)")
    plt.fill_between(
        data["steps"], 
        data["mean"] - data["std"], 
        data["mean"] + data["std"], 
        alpha=0.2
    )

plt.xlabel("Timesteps")
plt.ylabel("Reward")
plt.title("PPO Learning Curves on LunarLander (Continuous)")
plt.legend()
plt.grid(True)
plt.savefig("learning_curves.png")
plt.show()